## Introduction

Classic visualizations—line charts, maps, rankings—remain the trusted way to explore baby names: clear, rigorous, and widely understood. **We go further:** we *translate* the same patterns into forms that are equally faithful yet more intuitive and memorable, without giving up analytical rigour. Our three interactive views—temporal evolution (*Popularity vines*), regional variation (choropleth + *Regional compass*), and gender effects (*Boys vs girls per name*)—pair data science with visual storytelling. Vines trace a name's life cycle; a compass condenses geography into petals; silhouettes make gender ratios visible at a glance. Metaphor here is not decoration but a second channel of understanding, anchored by the same controls and computations as conventional charts. Our aim: data that can be *felt* as well as measured—readable, memorable, and lived.

# Visualizations 1.6, 2.1 & 3.3 — interactive implementation

Subset of the 9 final designs from `baby_names_9_viz_group_G_.ipynb`:

| Problem | Design |
|---|---|
| **1 — Temporal** | 1.6 Popularity vines |
| **2 — Regional** | 2.1 Choropleth map + 2.9 Regional compass |
| **3 — Gender** | 3.3 Boys vs girls per name |


In [1]:
import json
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output

alt.data_transformers.disable_max_rows()

# ── Load & clean data ──────────────────────────────────────────────────────
df = pd.read_csv('../data/dpt2020.csv', sep=';', dtype={'annais': str, 'dpt': str, 'nombre': str})
df = df[(df['annais'] != 'XXXX') & (df['preusuel'] != '_PRENOMS_RARES')]
df['annais'] = pd.to_numeric(df['annais'], errors='coerce')
df['nombre'] = pd.to_numeric(df['nombre'], errors='coerce')
df = df.dropna(subset=['annais', 'nombre'])
df['annais'] = df['annais'].astype(int)
df['decade'] = (df['annais'] // 10) * 10

nat = df.groupby(['preusuel', 'annais', 'sexe'])['nombre'].sum().reset_index()
nat['decade'] = (nat['annais'] // 10) * 10
nat_dec = nat.groupby(['preusuel', 'decade', 'sexe'])['nombre'].sum().reset_index()
nat_dec['rank'] = nat_dec.groupby(['decade', 'sexe'])['nombre'].rank(ascending=False, method='min').astype(int)
nat_dec['sex_label'] = nat_dec['sexe'].map({1: 'Male', 2: 'Female'})

pop_total = nat.groupby(['preusuel', 'annais'])['nombre'].sum().reset_index()
top50 = df.groupby('preusuel')['nombre'].sum().nlargest(50).index.tolist()
top20_map = top50[:20]

# Map / regional aggregates
total_dpt = df.groupby(['dpt', 'annais'])['nombre'].sum().reset_index()
total_dpt['decade'] = (total_dpt['annais'] // 10) * 10
total_dpt_dec = total_dpt.groupby(['dpt', 'decade'])['nombre'].sum().reset_index()
total_dpt_dec.columns = ['dpt', 'decade', 'total_births']

name_dpt = df.groupby(['preusuel', 'dpt', 'annais'])['nombre'].sum().reset_index()
name_dpt['decade'] = (name_dpt['annais'] // 10) * 10
name_dpt_dec = name_dpt.groupby(['preusuel', 'dpt', 'decade'])['nombre'].sum().reset_index()
name_dpt_dec = name_dpt_dec.merge(total_dpt_dec, on=['dpt', 'decade'], how='left')
name_dpt_dec['rate_per_10k'] = (name_dpt_dec['nombre'] / name_dpt_dec['total_births'] * 10000).round(2)
map_data = name_dpt_dec[name_dpt_dec['preusuel'].isin(top50)].copy()
decades_map = sorted(map_data['decade'].unique().tolist())

with open('../data/departements.geojson') as fgeo:
    geo = json.load(fgeo)
geo_inline = alt.InlineData(values=geo['features'], format=alt.DataFormat(type='json'))
dpt_names_full = {f['properties']['code']: f['properties']['nom'] for f in geo['features']}
dpt_names_full.update({
    '971': 'Guadeloupe', '972': 'Martinique', '973': 'Guyane',
    '974': 'La Réunion', '976': 'Mayotte', '20': 'Corse (ancien)',
})

region_map = {
    'Île-de-France': ['75', '77', '78', '91', '92', '93', '94', '95'],
    'Bretagne': ['22', '29', '35', '56'],
    'PACA': ['04', '05', '06', '13', '83', '84'],
    'Auvergne-RA': ['01', '03', '07', '15', '26', '38', '42', '43', '63', '69', '73', '74'],
    'Occitanie': ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82'],
    'Nouvelle-Aquit.': ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87'],
    'Grand-Est': ['08', '10', '51', '52', '54', '55', '57', '67', '68', '88'],
    'Normandie': ['14', '27', '50', '61', '76'],
    'Hauts-de-France': ['02', '59', '60', '62', '80'],
    'Pays-de-la-Loire': ['44', '49', '53', '72', '85'],
}
dpt_to_reg = {dpt: reg for reg, dpts in region_map.items() for dpt in dpts}
map_data['region'] = map_data['dpt'].map(dpt_to_reg)
reg_agg = (
    map_data.dropna(subset=['region'])
    .groupby(['preusuel', 'region', 'decade'])
    .agg(nombre=('nombre', 'sum'), total_births=('total_births', 'sum'))
    .reset_index()
)
reg_agg['rate_per_10k'] = (reg_agg['nombre'] / reg_agg['total_births'] * 10000).round(2)

# Gender aggregates
gender_nat = nat.groupby(['preusuel', 'sexe'])['nombre'].sum().reset_index()
g_pivot = gender_nat.pivot_table(index='preusuel', columns='sexe', values='nombre', fill_value=0).reset_index()
g_pivot.columns.name = None
g_pivot = g_pivot.rename(columns={1: 'male', 2: 'female'})
for col in ('male', 'female'):
    if col not in g_pivot.columns:
        g_pivot[col] = 0
g_pivot['total'] = g_pivot['male'] + g_pivot['female']
g_pivot['female_ratio'] = g_pivot['female'] / g_pivot['total']
g_pivot['mixedness'] = (g_pivot['female_ratio'] - 0.5).abs()

gender_dec = nat.groupby(['preusuel', 'sexe', 'decade'])['nombre'].sum().reset_index()
g_piv_dec = gender_dec.pivot_table(index=['preusuel', 'decade'], columns='sexe', values='nombre', fill_value=0).reset_index()
g_piv_dec.columns.name = None
g_piv_dec = g_piv_dec.rename(columns={1: 'male', 2: 'female'})
for col in ('male', 'female'):
    if col not in g_piv_dec.columns:
        g_piv_dec[col] = 0
g_piv_dec['total'] = g_piv_dec['male'] + g_piv_dec['female']
g_piv_dec['female_ratio'] = g_piv_dec['female'] / g_piv_dec['total']
g_piv_dec['gender_type'] = 'Mixed (both)'
g_piv_dec.loc[g_piv_dec['female_ratio'] > 0.8, 'gender_type'] = 'Mostly female'
g_piv_dec.loc[g_piv_dec['female_ratio'] < 0.2, 'gender_type'] = 'Mostly male'

unisex_names = g_pivot[(g_pivot['total'] >= 3000) & (g_pivot['mixedness'] <= 0.25)].sort_values('mixedness')['preusuel'].tolist()
mixed_top = g_pivot[g_pivot['total'] >= 5000].nsmallest(20, 'mixedness')['preusuel'].tolist()

print(f'{len(df):,} rows · {df["preusuel"].nunique():,} names · {df["annais"].min()}–{df["annais"].max()}')


3,773,985 rows · 16,226 names · 1900–2022


# Problem 1 — Temporal evolution

## 1.6 — Popularity vines

In [27]:
# Top-10 names on x-axis; decades on y-axis; one vertical vine per name; leaf marks per decade
LEAF_SHAPE = 'M0,-7 C3,-4 5,1 0,8 C-5,1 -3,-4 0,-7 Z'

def _leaf_angle(name, decade):
    # Stable pseudo-random tilt per leaf (roughly ±55°)
    h = hash((name, int(decade))) % 110
    return h - 55

def make_leaf_frieze(sex_label, decade_from=1900):
    sex_code = 2 if sex_label == 'Female' else 1
    period = nat_dec[(nat_dec['sexe'] == sex_code) & (nat_dec['decade'] >= decade_from)]
    name_totals = period.groupby('preusuel')['nombre'].sum().sort_values(ascending=False)
    x_sort = name_totals.nlargest(10).index.tolist()
    d = period[period['preusuel'].isin(x_sort)].copy()
    d['decade_label'] = d['decade'].astype(str) + 's'
    d['norm'] = d.groupby('decade')['nombre'].transform(lambda x: x / x.max())
    d['leaf_angle'] = [_leaf_angle(n, dec) for n, dec in zip(d['preusuel'], d['decade'])]

    decades = sorted(d['decade'].unique().tolist())
    decade_max = decades[-1]
    decade_min = decades[0]
    decade_scale = alt.Scale(reverse=True, domain=[decade_max, decade_min])
    decade_axis = alt.Axis(values=decades, labelExpr="datum.value + 's'")
    chart_height = max(420, min(720, 120 + len(decades) * 42))

    y_decade = alt.Y('decade:Q', title='Decade', scale=decade_scale, axis=decade_axis)

    x_name = alt.X('preusuel:N', sort=x_sort, title='Name (top 10, by total births)')

    vines = alt.Chart(d).mark_line(color='#689f38', strokeWidth=2, opacity=0.75).encode(
        x=x_name,
        y=y_decade,
        order=alt.Order('decade:Q'),
        detail='preusuel:N',
    )
    leaves = alt.Chart(d).mark_point(
        filled=True, shape=LEAF_SHAPE, stroke='#1b5e20', strokeWidth=0.35,
    ).encode(
        x=x_name,
        y=y_decade,
        angle=alt.Angle('leaf_angle:Q', scale=None),
        size=alt.Size(
            'nombre:Q',
            scale=alt.Scale(range=[25, 500], type='sqrt'),
            legend=None,
        ),
        color=alt.Color(
            'norm:Q',
            scale=alt.Scale(scheme='greens', domain=[0, 1]),
            title='Relative popularity (per decade)',
        ),
        tooltip=[
            'preusuel:N',
            alt.Tooltip('decade_label:N', title='Decade'),
            alt.Tooltip('nombre:Q', title='Births (decade)', format=','),
            alt.Tooltip('norm:Q', title='Relative popularity (decade)', format='.2f'),
        ],
    )
    return (vines + leaves).properties(
        width=820, height=chart_height,
        title=alt.Title(
            f'Popularity vines — top 10 names ({sex_label})',
            subtitle=(
                f'Each column = one name vine along decades ({decade_from}s–{decade_max}s). '
                'Leaf size = births in the decade; darker green = higher relative popularity within that decade (among the top 10).'
            ),
        ),
    )

leaf_decades = list(range(1900, int(df['annais'].max()) + 1, 10))
leaf_sex = widgets.RadioButtons(
    options=['Female', 'Male'], value='Female', description='Sex:',
)
leaf_from = widgets.IntSlider(
    value=1900, min=leaf_decades[0], max=leaf_decades[-1], step=10,
    description='From decade:', continuous_update=False,
)
leaf_out = widgets.Output()

def on_leaf_change(_):
    with leaf_out:
        clear_output(wait=True)
        display(make_leaf_frieze(leaf_sex.value, leaf_from.value))

leaf_sex.observe(on_leaf_change, names='value')
leaf_from.observe(on_leaf_change, names='value')
display(widgets.HBox([leaf_sex, leaf_from]), leaf_out)
on_leaf_change(None)


Output()

### Comments — 1.6 Popularity vines

**Question (Problem 1):** How do French baby names evolve between 1900 and 2022?

**Reading the chart:** Top 10 names (x) × decades (y). Each **vine** tracks one name; **leaf size** = births, **green shade** = relative rank within that decade. Growing then fading leaves suggest a name's life cycle; side-by-side columns show generational turnover. Because leaf size encodes absolute counts, the vines also echo broader population change—baby booms, wartime dips, and long-term declines in birth rates show up alongside each name's rise and fall. Sex and start-decade controls compare trends across groups and periods.

**Design:** Botanical metaphor (vine + tilted leaves) turns a time series into an intuitive frieze rather than a standard line chart.

**Strengths:** Memorable storytelling, ten life cycles in one view, tooltips keep exact counts.

**Limitations:** Less precise than a rank chart; limited to the top 10 names.


# Problem 2 — Regional effects

## 2.1 — Choropleth map & regional compass (2.1 + 2.9)

In [39]:
# 2.9 — Regional compass (macro-regions on geographic bearings)
REGION_TO_FLOWER = {
    'Hauts-de-France': 'Nord', 'Normandie': 'Nord',
    'Bretagne': 'Ouest', 'Pays-de-la-Loire': 'Ouest',
    'Île-de-France': 'IDF', 'Grand-Est': 'Est',
    'Nouvelle-Aquit.': 'S-O', 'Occitanie': 'S-O',
    'PACA': 'S-E', 'Auvergne-RA': 'Centre',
}
CORSE_DPT = {'2A', '2B', '20'}
FLOWER_ORDER = ['Nord', 'Ouest', 'IDF', 'Est', 'S-O', 'S-E', 'Corse', 'Centre']
# Radians from East, counter-clockwise (π/2 = North, y-up)
COMPASS_ANGLE = {
    'Nord': np.pi / 2,
    'Ouest': np.pi,
    'IDF': np.pi / 2 - np.pi / 8,
    'Est': 0,
    'S-O': -3 * np.pi / 4,
    'S-E': -np.pi / 4,
    'Corse': -np.pi / 6,
    'Centre': -np.pi / 2,
}

def _flower_zone(row):
    if row['dpt'] in CORSE_DPT:
        return 'Corse'
    reg = row.get('region')
    if pd.isna(reg):
        reg = dpt_to_reg.get(row['dpt'])
    return REGION_TO_FLOWER.get(reg)

def _flower_data(name, decade):
    sub = df[(df['preusuel'] == name) & (df['decade'] == decade)].groupby('dpt')['nombre'].sum().reset_index()
    tot = df[df['decade'] == decade].groupby('dpt')['nombre'].sum().reset_index(name='total_births')
    sub = sub.merge(tot, on='dpt', how='left')
    sub['rate_per_10k'] = (sub['nombre'] / sub['total_births'] * 10000).fillna(0)
    sub['region'] = sub['dpt'].map(dpt_to_reg)
    sub['flower'] = sub.apply(_flower_zone, axis=1)
    sub = sub.dropna(subset=['flower'])
    rows = []
    for flower, g in sub.groupby('flower'):
        rate = np.average(g['rate_per_10k'], weights=g['nombre'].clip(lower=1))
        rows.append({'flower': flower, 'rate_per_10k': rate})
    if not rows:
        return pd.DataFrame({'flower': FLOWER_ORDER, 'rate_per_10k': [0.0] * len(FLOWER_ORDER)})
    return pd.DataFrame(rows).set_index('flower').reindex(FLOWER_ORDER).fillna(0).reset_index()

def make_choropleth(name, decade):
    filtered = map_data[(map_data['preusuel'] == name) & (map_data['decade'] == decade)][['dpt', 'rate_per_10k', 'nombre']]
    return alt.Chart(geo_inline).mark_geoshape(stroke='white', strokeWidth=0.5).transform_lookup(
        lookup='properties.code',
        from_=alt.LookupData(alt.InlineData(values=filtered.to_dict(orient='records')), 'dpt', ['rate_per_10k', 'nombre']),
    ).encode(
        color=alt.Color('rate_per_10k:Q', scale=alt.Scale(scheme='blues', domainMin=0), title='Births / 10k'),
        tooltip=[alt.Tooltip('properties.nom:N', title='Department'),
                 alt.Tooltip('rate_per_10k:Q', title='Rate / 10k', format='.1f'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')],
    ).project('mercator').properties(
        width=580, height=520,
        title=alt.Title('Department map', subtitle='Darker blue = higher births / 10k.'),
    )

def make_regional_compass(name, decade):
    d = _flower_data(name, decade)
    d['angle'] = d['flower'].map(COMPASS_ANGLE)
    r0 = 0.38
    scale = 2.6 / max(d['rate_per_10k'].max(), 0.1)
    d['x'] = r0 * np.cos(d['angle'])
    d['y'] = r0 * np.sin(d['angle'])
    r_end = r0 + d['rate_per_10k'] * scale
    d['x2'] = r_end * np.cos(d['angle'])
    d['y2'] = r_end * np.sin(d['angle'])
    d_tip = d.assign(x=d['x2'], y=d['y2'])
    d_lbl = d.assign(x=(r_end + 0.24) * np.cos(d['angle']), y=(r_end + 0.24) * np.sin(d['angle']))
    lim = r0 + d['rate_per_10k'].max() * scale + 0.62
    sc = alt.Scale(domain=[-lim, lim])
    xy = dict(scale=sc, axis=None)

    ring_t = np.linspace(0, 2 * np.pi, 72)
    ring = pd.DataFrame({'x': r0 * 0.92 * np.cos(ring_t), 'y': r0 * 0.92 * np.sin(ring_t)})
    ring_chart = alt.Chart(ring).mark_line(color='#bbb', strokeWidth=1.5).encode(
        x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy),
    )
    spokes = alt.Chart(d).mark_rule(strokeWidth=11, color='#7cb342', opacity=0.92).encode(
        x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy), x2='x2:Q', y2='y2:Q',
    )
    tips = alt.Chart(d_tip).mark_circle(size=90, color='#2e7d32', stroke='black', strokeWidth=1.2).encode(
        x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy),
        tooltip=['flower:N', alt.Tooltip('rate_per_10k:Q', title='Births / 10k', format='.1f')],
    )
    zone_labels = alt.Chart(d_lbl).mark_text(fontSize=10, fontWeight='bold').encode(
        x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy), text='flower:N',
    )
    hub = alt.Chart(pd.DataFrame({'x': [0], 'y': [0]})).mark_circle(
        size=1200, color='white', stroke='black', strokeWidth=2,
    ).encode(x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy))
    hub_text = alt.Chart(pd.DataFrame({'x': [0], 'y': [0], 't': [name]})).mark_text(
        fontSize=10, fontWeight='bold',
    ).encode(x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy), text='t:N')
    note = alt.Chart(pd.DataFrame({
        'x': [0], 'y': [-lim * 0.92],
        't': ['Spoke length = births per 10k in region'],
    })).mark_text(fontSize=9, color='#555', align='center').encode(
        x=alt.X('x:Q', **xy), y=alt.Y('y:Q', **xy), text='t:N',
    )

    return (ring_chart + spokes + tips + zone_labels + hub + hub_text + note).resolve_scale(
        x='shared', y='shared',
    ).properties(
        width=480, height=520,
        title=alt.Title('Regional compass', subtitle='Direction = geography'),
    )

def make_regional_view(name, decade):
    return alt.hconcat(
        make_choropleth(name, decade),
        make_regional_compass(name, decade),
        spacing=0,
    ).resolve_scale(color='independent').properties(
        title=alt.Title(f'Regional popularity — {name} ({decade}s)',
                        subtitle='Left: department choropleth · Right: regional compass.'),
    ).configure_view(stroke=None).configure_axis(
        labels=False, ticks=False, title=None, domain=False, grid=False,
    )

map_name = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
map_dec = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
map_out = widgets.Output()

def on_map_change(_):
    with map_out:
        clear_output(wait=True)
        display(make_regional_view(map_name.value, map_dec.value))

map_name.observe(on_map_change, names='value')
map_dec.observe(on_map_change, names='value')
display(widgets.HBox([map_name, map_dec]), map_out)
on_map_change(None)


Output()

### Comments — 2.1 Choropleth map & 2.9 Regional compass

**Question (Problem 2):** Are certain names more popular in specific regions or departments?

**Reading the chart:** The **choropleth** (left) colours each department by births per 10,000 — darker blue = higher relative popularity for the selected name and decade. The **regional compass** (right) aggregates departments into macro-zones (Nord, Ouest, IDF, etc.); spoke direction follows geography, spoke length encodes the same normalized rate. Together they give fine-grained location and a compact regional profile. Name and decade dropdowns let you compare how a name’s geography shifts over time.

**Design:** Classic map plus an abstract compass metaphor — precision at department level paired with a synthetic, comparable regional summary.

**Strengths:** Intuitive geography on the map; compass makes cross-name and cross-decade comparison fast; normalized rates avoid population bias; tooltips keep exact values.

**Limitations:** One name at a time; compass loses department detail; overseas territories and Corsa grouping simplify real geography.

# Problem 3 — Gender effects

## 3.3 — Boys vs girls per name

In [26]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from io import BytesIO
from IPython.display import Image, display, clear_output
from matplotlib.patches import PathPatch, Circle, Arc, Patch
from matplotlib.path import Path

if 'g_piv_dec' not in globals() or 'decades_map' not in globals():
    raise RuntimeError('Run the data-loading cell first (cell above Problem 1).')

GRID_ROWS, GRID_COLS = 5, 5
MIN_BIRTHS_DECADE = 50
BOY_COLOR, GIRL_COLOR = '#5b9bd5', '#f4b6c8'
BOY_TEXT, GIRL_TEXT = '#2e6eb3', '#c44b7a'
BABY_YMIN, BABY_YMAX = 0.16, 0.94

def _baby_silhouette_path():
    """Sitting-baby outline matching the hand-drawn schematic."""
    verts = [
        (0.50, 0.94), (0.62, 0.92), (0.70, 0.84), (0.72, 0.74), (0.66, 0.66),
        (0.78, 0.60), (0.82, 0.48), (0.76, 0.38), (0.66, 0.32), (0.58, 0.22),
        (0.50, 0.16), (0.42, 0.22), (0.34, 0.32), (0.24, 0.38), (0.18, 0.48),
        (0.22, 0.60), (0.34, 0.66), (0.28, 0.74), (0.30, 0.84), (0.38, 0.92),
        (0.50, 0.94),
    ]
    codes = [Path.MOVETO] + [Path.LINETO] * (len(verts) - 2) + [Path.CLOSEPOLY]
    return Path(verts, codes)

def _paint_baby_fill(ax, path, male_pct, female_pct, raster=200):
    """Raster mask fill — reliable blue/pink split inside the silhouette."""
    split = BABY_YMIN + (BABY_YMAX - BABY_YMIN) * (female_pct / 100.0)
    n = raster
    xs = np.linspace(0, 1, n)
    ys = np.linspace(0, 1, n)
    X, Y = np.meshgrid(xs, ys)
    inside = path.contains_points(np.column_stack([X.ravel(), Y.ravel()])).reshape(n, n)
    boy = np.array(mcolors.to_rgb(BOY_COLOR))
    girl = np.array(mcolors.to_rgb(GIRL_COLOR))
    rgba = np.zeros((n, n, 4))
    is_boy = inside & (Y >= split)
    is_girl = inside & (Y < split)
    rgba[is_boy, :3] = boy
    rgba[is_boy, 3] = 1
    rgba[is_girl, :3] = girl
    rgba[is_girl, 3] = 1
    ax.imshow(rgba, extent=[0, 1, 0, 1], origin='lower', aspect='equal',
              zorder=1, interpolation='nearest')

def _draw_one_baby(ax, name, male_pct, female_pct, compact=False):
    ax.set_xlim(0.13, 0.87)
    ax.set_ylim(0.13, 0.97)
    ax.set_aspect('equal')
    ax.axis('off')

    path = _baby_silhouette_path()
    _paint_baby_fill(ax, path, male_pct, female_pct, raster=160 if compact else 200)

    outline_lw = 1.5 if compact else 2.6
    face_lw = 1.1 if compact else 2.2
    eye_r = 0.017 if compact else 0.022
    ax.add_patch(PathPatch(path, facecolor='none', edgecolor='black', linewidth=outline_lw, zorder=2))
    ax.add_patch(Circle((0.43, 0.79), eye_r, color='black', zorder=3))
    ax.add_patch(Circle((0.57, 0.79), eye_r, color='black', zorder=3))
    ax.add_patch(Arc((0.50, 0.73), 0.12, 0.07, angle=0, theta1=200, theta2=340,
                     linewidth=face_lw, edgecolor='black', fill=False, zorder=3))
    ax.add_patch(Arc((0.63, 0.91), 0.07, 0.05, angle=25, theta1=210, theta2=330,
                     linewidth=face_lw, edgecolor='black', fill=False, zorder=3))

    label = name.title() if len(name) <= 10 else name.title()[:9] + '…'
    ax.text(0.5, 1.02, label, ha='center', va='bottom',
            fontsize=8 if compact else 16, fontweight='bold', transform=ax.transAxes)
    if not compact:
        ax.text(0.34, -0.04, f'{male_pct:.0f}%', ha='center', color=BOY_TEXT, fontsize=14,
                fontweight='bold', transform=ax.transAxes)
        ax.text(0.66, -0.04, f'{female_pct:.0f}%', ha='center', color=GIRL_TEXT, fontsize=14,
                fontweight='bold', transform=ax.transAxes)

def _ratio_row(name, decade):
    d = g_piv_dec[(g_piv_dec['preusuel'] == name) & (g_piv_dec['decade'] == decade)]
    if d.empty:
        return 50.0, 50.0
    row = d.iloc[0]
    male_pct = (1 - row['female_ratio']) * 100
    female_pct = row['female_ratio'] * 100
    return male_pct, female_pct

def _fmt_ratio_band(lo, hi):
    if lo == hi:
        return f'{lo:.0%}'
    return f'{lo:.0%}–{hi:.0%}'

def _ratio_grid(decade):
    """5×5 matrix: even 20%-wide female-ratio bands; top 5 names per band by births."""
    d = g_piv_dec[(g_piv_dec['decade'] == decade) & (g_piv_dec['total'] >= MIN_BIRTHS_DECADE)].copy()
    grid, bin_ranges, used = [], [], set()
    for bin_idx in range(GRID_ROWS):
        lo, hi = bin_idx / GRID_ROWS, (bin_idx + 1) / GRID_ROWS
        center = (lo + hi) / 2
        bin_ranges.append((lo, hi))
        pool = d[~d['preusuel'].isin(used)].copy()
        in_band = pool[(pool['female_ratio'] >= lo) & (pool['female_ratio'] <= hi)]
        picked = in_band.sort_values(['total', 'preusuel'], ascending=[False, True])['preusuel'].tolist()
        if len(picked) < GRID_COLS:
            rest = pool[~pool['preusuel'].isin(picked)].copy()
            rest['dist'] = (rest['female_ratio'] - center).abs()
            extra = (
                rest.sort_values(['dist', 'total', 'preusuel'], ascending=[True, False, True])
                .head(GRID_COLS - len(picked))['preusuel'].tolist()
            )
            picked += extra
        picked = picked[:GRID_COLS]
        used.update(picked)
        grid.append(picked + [None] * (GRID_COLS - len(picked)))
    return grid, bin_ranges

def make_baby_ratio_sketch(decade):
    grid, bin_ranges = _ratio_grid(decade)
    fig, axes = plt.subplots(GRID_ROWS, GRID_COLS, figsize=(14, 12))
    fig.subplots_adjust(left=0.10, right=0.92, top=0.90, bottom=0.06, wspace=0.10, hspace=0.38)

    for row_idx, row_names in enumerate(grid):
        for col_idx, name in enumerate(row_names):
            ax = axes[row_idx, col_idx]
            if name is None:
                ax.axis('off')
                continue
            male_pct, female_pct = _ratio_row(name, decade)
            _draw_one_baby(ax, name, male_pct, female_pct, compact=True)

    fig.text(
        0.04, 0.5, 'Female ratio ↑', va='center', ha='center', rotation='vertical',
        fontsize=12, fontweight='bold',
    )
    for i in range(GRID_ROWS):
        lo, hi = bin_ranges[i]
        y = 0.06 + (GRID_ROWS - 0.5 - i) / GRID_ROWS * 0.84
        fig.text(0.07, y, _fmt_ratio_band(lo, hi), va='center', ha='right', fontsize=8, color='#555')

    fig.suptitle(
        f'Who wears this name? — boys vs girls ({decade}s)',
        fontsize=15, fontweight='bold', y=0.97,
    )
    fig.text(
        0.5, 0.93,
        'Blue = boys, pink = girls. Rows group names by gender balance; columns rank by popularity.',
        ha='center', fontsize=10, color='#444',
    )
    fig.legend(
        handles=[Patch(facecolor=BOY_COLOR, label='Garçons'),
                 Patch(facecolor=GIRL_COLOR, label='Filles')],
        loc='upper right', bbox_to_anchor=(0.99, 0.96), frameon=True, fontsize=10,
    )
    return fig

def _baby_figure_image(decade):
    plt.close('all')
    fig = make_baby_ratio_sketch(decade)
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    return Image(data=buf.getvalue())

globals().pop('_baby_ui', None)  # drop stale widget cache from earlier versions

baby_dec = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
baby_out = widgets.Output()

def _on_baby_change(_):
    with baby_out:
        clear_output(wait=True)
        display(_baby_figure_image(baby_dec.value))

display(widgets.VBox([baby_dec, baby_out]))
with baby_out:
    display(_baby_figure_image(baby_dec.value))
baby_dec.observe(_on_baby_change, names='value')


### Comments — 3.3 Boys vs girls per name

**Question (Problem 3):** Are names mostly given to girls, boys, or both — and how does that balance vary?

**Reading the chart:** A **5×5 grid** of baby silhouettes for the selected decade. Each icon is split **blue (boys) / pink (girls)** by the female ratio; **rows** group names into 20%-wide gender-balance bands (bottom = mostly male, top = mostly female); **columns** rank names by popularity within each band. Scanning a row reveals typical names at a given gender mix; moving up the grid shows the shift toward female-dominated names.

**Design:** Icon-based gender encoding turns abstract ratios into an immediately readable visual — no axes or percentages required to grasp the idea.

**Strengths:** Very accessible for non-experts; 25 examples per decade cover the full gender spectrum; decade selector shows how naming conventions evolve.

**Limitations:** Approximate rather than exact (compact icons); only 25 names per view; band assignment can place borderline names in adjacent rows.